# From OSI to SLayer

**TL;DR:** SLayer stores an [OSI](https://open-semantic-interchange.org/) (Open Semantic Interchange) semantic-model document natively, as written: its `datasets`, `relationships`, and `metrics` are queryable as they are, with nothing converted. This notebook imports a small retail OSI document end-to-end and answers five questions, checking each answer against gold SQL.

It runs as four explicit steps (everything lands in a gitignored `.cache/` next to this notebook):

1. **Build the data** — create a tiny retail DuckDB (orders, customers, products, regions) with deterministic rows.
2. **Reference answers** — run the gold SQL *up front*, before the importer touches the file.
3. **Import** — store `shop.osi.yaml` as a native semantic-model document with `import_osi`.
4. **Query** — point a SLayer client at the stored document and verify against gold.

> Unlike the [dbt MetricFlow demo](../11_dbt_metricflow/dbt_metricflow_nb.ipynb), this one is **fully offline** — no network access is needed at any point.

The committed OSI config ([`shop.osi.yaml`](shop.osi.yaml)) exercises `ai_context` metadata, `nonportable_extensions`, relationships (including a multi-hop chain), and dataset-scoped (including a `FILTER`) and cross-dataset / multi-hop model-scoped metrics, in the current flat Ossie shape.

## Step 1 — Build the demo database

Create the retail DuckDB the OSI document binds to. The import itself never touches the database; queries do.

In [1]:
import os
import sys

import pandas as pd

# The setup helper lives next to this notebook.
sys.path.insert(0, os.getcwd())

from setup_osi import (
    build_shop_duckdb,
    import_osi_document,
    compute_gold,
    OSI_CONFIG,
    DB_PATH,
    DATASETS_DIR,
    DATASOURCE_NAME,
)
from slayer.async_utils import run_sync
from slayer.client.slayer_client import SlayerClient
from slayer.storage.yaml_storage import YAMLStorage

db_path = build_shop_duckdb(DB_PATH)
print(f"Built retail DuckDB '{db_path.name}' with orders, customers, products, regions.")

Built retail DuckDB 'shop.duckdb' with orders, customers, products, regions.


## Step 2 — Reference answers (gold SQL)

We run the gold SQL **first**, up front, and stash the expected numbers.

Why up front? SLayer's queries open a read-write SQLAlchemy engine on the DuckDB file — and DuckDB won't let a second raw connection share the file under a different configuration. So every gold query runs *before* SLayer touches the file.

In [2]:
# The reference answers live in setup_osi.compute_gold — shared with the
# agent-workflow notebook so both demos check against identical numbers.
GOLD = compute_gold(DB_PATH)

print("gold total amount :", GOLD["total"])
print("gold aov          :", GOLD["aov"])
print("gold cust_reach   :", GOLD["cust_reach"])
print("gold rev_plus_pop :", GOLD["rev_plus_pop"])
pd.DataFrame(GOLD["by_region"])

gold total amount : 1400.0
gold aov          : 233.33333333333334
gold cust_reach   : 466.6666666666667
gold rev_plus_pop : 4400.0


,region,amount
0,North,750.0
1,South,650.0


## Step 3 — Import the OSI config

This is the heart of the demo. `import_osi_document` registers the DuckDB as a SLayer datasource and stores `shop.osi.yaml` with [`import_osi`](../../osi/osi_import.md), its text unchanged:

- each OSI **dataset** is a SLayer dataset,
- each **relationship** is a many-to-one relationship on its `from` dataset,
- each dataset **metric** is a metric of its dataset, and each top-level metric a **model-scoped** metric, queried by its bare name.

The import report lists anything SLayer cannot evaluate yet — here, nothing.

In [3]:
result = import_osi_document(OSI_CONFIG, DATASETS_DIR, DB_PATH)

print("Stored semantic models:", result.stored)
print(result.render_report())

Stored semantic models: ['shop']
No import issues.


## Step 4 — The imported metadata survived

OSI's `ai_context` and `nonportable_extensions` don't just decorate the source file — the importer carries them onto the SLayer entities so agents can read them. Below, the `orders.amount` field keeps its label, its `ai_context` instructions as a description, and both blocks in `meta`; the `orders -> customers` relationship keeps its OSI relationship `ai_context` as a description.

In [4]:
storage = YAMLStorage(base_dir=str(DATASETS_DIR))
orders = run_sync(storage.get_dataset("orders", data_source=DATASOURCE_NAME))

amount = next(f for f in orders.fields if f.name == "amount")
print("orders.amount.label       :", amount.label)
print("orders.amount.description :", amount.description)
print("orders.amount.meta        :", amount.meta)
print()
for r in orders.relationships:
    print(f"relationship orders -> {r.to}: {r.description!r}")
print()
print("metrics on orders:", [m.name for m in orders.metrics])
print("model-scoped metrics:", [m.name for m in run_sync(storage.list_model_scoped_metrics(DATASOURCE_NAME))])

orders.amount.label       : Order amount
orders.amount.description : None
orders.amount.meta        : None

relationship orders -> customers: None
relationship orders -> products: None

metrics on orders: ['total_amount', 'completed_amount', 'order_count', 'aov', 'revenue_line']
model-scoped metrics: ['cust_reach', 'rev_plus_pop']


## Query the imported datasets

Point a SLayer client at the stored document. Every query below is plain SLayer JSON.

In [5]:
client = SlayerClient(storage=YAMLStorage(base_dir=str(DATASETS_DIR)))


def approx(a, b, tol=1e-9):
    return abs(a - b) < tol


def show(got, exp):
    """Print a SLayer scalar answer next to its gold value."""
    print("SLayer:", got, " gold:", exp)

### Query 1 — total order value

A simple dataset-scoped metric: `total_amount` is `SUM(amount)` on the `orders` dataset, read as written.

In [6]:
q1 = {"population": "orders", "metrics": ["orders.total_amount"]}
rows = client.query_sync(q1).data
print("SLayer:", rows)

assert approx(rows[0]["orders.total_amount"], GOLD["total"])
print(f"OK — SLayer and gold agree: {GOLD['total']}")

SLayer: [{'orders.total_amount': 1400.0}]
OK — SLayer and gold agree: 1400.0


### Query 2 — total by region (multi-hop relationship path)

`orders.total_amount` grouped by `customers.regions.name` walks two of the document's relationships — `orders -> customers -> regions` — with no SQL written by hand. The result column is keyed by the canonical path, which spells each hop by its relationship name: `customers.customers_to_regions.name`.

In [7]:
q2 = {
    "population": "orders",
    "metrics": ["orders.total_amount"],
    "dimensions": ["customers.regions.name"],
    "order": [{"field": "customers.regions.name"}],
}
rows = client.query_sync(q2).data
display(pd.DataFrame(rows))

slayer_pairs = [(r["customers.customers_to_regions.name"], r["orders.total_amount"]) for r in rows]
gold_pairs = [(g["region"], g["amount"]) for g in GOLD["by_region"]]
assert slayer_pairs == gold_pairs, f"{slayer_pairs} != {gold_pairs}"
print(f"OK — SLayer and gold agree on {len(slayer_pairs)} region(s)")

,customers.customers_to_regions.name,orders.total_amount
0,North,750.0
1,South,650.0


OK — SLayer and gold agree on 2 region(s)


### Query 3 — average order value (derived metric)

`aov` came from the OSI expression `SUM(amount) / COUNT(*)` — a **derived** metric over two aggregates, read as written.

In [8]:
q3 = {"population": "orders", "metrics": ["orders.aov"]}
rows = client.query_sync(q3).data
show(rows[0]["orders.aov"], GOLD["aov"])

assert approx(rows[0]["orders.aov"], GOLD["aov"])
print("OK — average order value matches gold")

SLayer: 233.33333333333334  gold: 233.33333333333334
OK — average order value matches gold


### Query 4 — amount per distinct customer (cross-dataset metric)

`cust_reach` = `SUM(orders.amount) / COUNT(DISTINCT customers.customer_id)`. It is a top-level (model-scoped) metric, so it belongs to no dataset and is queried by its bare name, with no population: each aggregate is evaluated over the rows of the one dataset it reads.

In [9]:
q4 = {"metrics": ["cust_reach"]}
rows = client.query_sync(q4).data
show(rows[0]["cust_reach"], GOLD["cust_reach"])

assert approx(rows[0]["cust_reach"], GOLD["cust_reach"])
print("OK — amount per distinct customer matches gold")

SLayer: 466.6666666666667  gold: 466.6666666666667
OK — amount per distinct customer matches gold


### Query 5 — amount plus region population (multi-hop metric)

`rev_plus_pop` = `SUM(orders.amount) + SUM(regions.population)` reaches `regions` two relationships away (`orders -> customers -> regions`).

It is model-scoped too. The subtle part: SLayer aggregates `regions.population` **at the regions grain** (each region counted once), so adding this metric can't multiply the order rows. A naive five-way join would instead add a region's population *once per order* in that region and badly over-count. Our gold mirrors SLayer's isolation — summing population over the distinct regions the orders reach — and the two agree.

In [10]:
q5 = {"metrics": ["rev_plus_pop"]}
rows = client.query_sync(q5).data
show(rows[0]["rev_plus_pop"], GOLD["rev_plus_pop"])

assert approx(rows[0]["rev_plus_pop"], GOLD["rev_plus_pop"])
print("OK — SLayer isolates the joined aggregate; both agree at", GOLD["rev_plus_pop"])

SLayer: 4400.0  gold: 4400.0
OK — SLayer isolates the joined aggregate; both agree at 4400.0


## Recap

Starting from an OSI document we did not write, SLayer:

- stored it as written, with no conversion and no database access at import,
- queried each `dataset` and walked its `relationships` (including multi-hop),
- evaluated dataset metrics and model-scoped `metrics` by their bare names,
- and surfaced `ai_context` / `nonportable_extensions` as descriptions and `meta` for agents to read.

Every answer matched gold SQL — including the multi-hop metric, where SLayer's sub-query isolation kept the added aggregate from fanning out the order rows.

### Further reading

- [Importing OSI configs](../../osi/osi_import.md) — the import reference: what is stored, upgraded, refused and reported.
- The one-liner this notebook mirrors: `slayer import-osi shop.osi.yaml --datasource shop_osi` (against a registered datasource).